In [1]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_excel("Downloads/hantavirus_structured.xlsx")

In [4]:
data.head()

,text,label,source,category,difficulty
0,Hantavirus spreads through rodent urine and dr...,real,WHO,transmission,easy
1,Early symptoms include fever and muscle pain,real,CDC,symptoms,easy
2,There is no specific cure but supportive care ...,real,WHO,diagnosis,medium
3,Early diagnosis of hantavirus infection is cha...,real,WHO,diagnosis,medium
4,Early symptoms are similar to influenza and CO...,real,WHO,symptoms,easy


In [5]:
data.describe()

,text,label,source,category,difficulty
count,49,49,49,49,49
unique,49,2,13,6,3
top,Hantavirus spreads through rodent urine and dr...,real,Twitter,general,medium
freq,1,25,7,18,26


In [10]:
data.shape

(49, 5)

In [13]:
data.columns

Index(['text', 'label', 'source', 'category', 'difficulty'], dtype='object')

In [16]:
data['text'].value_counts()

text
Hantavirus spreads through rodent urine and droppings                                                1
Early symptoms include fever and muscle pain                                                         1
There is no specific cure but supportive care helps                                                  1
Early diagnosis of hantavirus infection is challenging                                               1
Early symptoms are similar to influenza and COVID-19                                                 1
Hantavirus symptoms can resemble dengue or sepsis                                                    1
Ribavirin may be a drug for HPS and HFRS but its effectiveness remains unknown                       1
HFRS is due to Old World hantaviruses while HPS is due to New World hantaviruses                     1
 Diagnosis is based on blood tests, typically serology                                               1
both form of diseases include low platelets and leaky blood vessels 

In [17]:
data['label'].value_counts()

label
real    25
fake    24
Name: count, dtype: int64

In [18]:
data['source'].value_counts()

source
Twitter     7
FB          6
YT          6
NIH         6
WHO         5
IG          5
CMI         4
CDC         2
CLA         2
EMJ         2
DOHW        2
BBC         1
Frontier    1
Name: count, dtype: int64

In [19]:
data['difficulty'].value_counts()

difficulty
medium    26
hard      12
easy      11
Name: count, dtype: int64

In [20]:
data['category'].value_counts()

category
general         18
treatment       10
prevention       6
transmission     6
symptoms         5
diagnosis        4
Name: count, dtype: int64

In [23]:
import re
import string

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["clean_text"] = data["text"].apply(clean_text)

data[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,Hantavirus spreads through rodent urine and dr...,hantavirus spreads through rodent urine and dr...,real
1,Early symptoms include fever and muscle pain,early symptoms include fever and muscle pain,real
2,There is no specific cure but supportive care ...,there is no specific cure but supportive care ...,real
3,Early diagnosis of hantavirus infection is cha...,early diagnosis of hantavirus infection is cha...,real
4,Early symptoms are similar to influenza and CO...,early symptoms are similar to influenza and covid,real


In [24]:
data.to_excel("hantavirus_cleaned.xlsx", index=False)

In [25]:
data.head()

,text,label,source,category,difficulty,clean_text
0,Hantavirus spreads through rodent urine and dr...,real,WHO,transmission,easy,hantavirus spreads through rodent urine and dr...
1,Early symptoms include fever and muscle pain,real,CDC,symptoms,easy,early symptoms include fever and muscle pain
2,There is no specific cure but supportive care ...,real,WHO,diagnosis,medium,there is no specific cure but supportive care ...
3,Early diagnosis of hantavirus infection is cha...,real,WHO,diagnosis,medium,early diagnosis of hantavirus infection is cha...
4,Early symptoms are similar to influenza and CO...,real,WHO,symptoms,easy,early symptoms are similar to influenza and covid


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
tfidf = TfidfVectorizer(stop_words='english')

In [29]:
X = tfidf.fit_transform(data["clean_text"])

In [31]:
y = data["label"]

In [32]:
print(X.shape)

(49, 194)


In [33]:
print(tfidf.get_feature_names_out())

['active' 'adverse' 'africa' 'air' 'americas' 'andes' 'antimalaria'
 'areas' 'asia' 'available' 'barely' 'based' 'bivalent' 'bleeeding'
 'blood' 'burning' 'campsite' 'cardiopulmonary' 'care' 'carry' 'catch'
 'caused' 'causes' 'challenging' 'chiefly' 'china' 'chloroquine' 'cities'
 'claim' 'cleaning' 'completely' 'contact' 'contagious' 'covid' 'creation'
 'cure' 'days' 'death' 'deaths' 'deep' 'dengue' 'diagnosis' 'disappear'
 'diseases' 'dobravabelgrade' 'drinking' 'droppings' 'drug' 'early'
 'effect' 'effective' 'effectiveness' 'eliminate' 'epidemica' 'europe'
 'exist' 'exists' 'fatality' 'fever' 'foods' 'forests' 'form'
 'garlicbased' 'given' 'governments' 'hantavirus' 'hantaviruses' 'hcps'
 'helps' 'hemorrhagic' 'herbal' 'herbs' 'hfrs' 'hiding' 'high' 'highly'
 'hoax' 'home' 'hospitalization' 'hot' 'hour' 'hours' 'hps' 'immunity'
 'inactivated' 'incense' 'include' 'indoors' 'infection' 'infections'
 'influenza' 'inhalation' 'internal' 'isolated' 'ivermectin' 'korea'
 'laboratory' 'le

In [35]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.4

Classification Report:
               precision    recall  f1-score   support

           0       0.40      0.40      0.40         5
           1       0.40      0.40      0.40         5

    accuracy                           0.40        10
   macro avg       0.40      0.40      0.40        10
weighted avg       0.40      0.40      0.40        10


Confusion Matrix:
 [[2 3]
 [3 2]]


In [52]:
sample = ["Hantavirus can only be supportive care "]

sample_clean = [clean_text(sample[0])]

sample_vector = tfidf.transform(sample_clean)

prediction = model.predict(sample_vector)

print(encoder.inverse_transform(prediction)[0])

real
